# LTA Road Risk Index — Unsupervised Deep Learning Pipeline

**Goal:** Dynamically compute a per-road-segment **Risk Index (0–100)** from live LTA DataMall APIs  
using a **Feedforward Autoencoder** (anomaly-detection-based) with domain-knowledge multipliers.

---

## Architecture Overview
```
LTA APIs (every 2–5 min)
  ├── Traffic Speed Bands  →  speed features per LinkID
  ├── Traffic Incidents    →  incident proximity & type
  ├── Faulty Traffic Lights→  junction safety signal
  ├── VMS / EMAS           →  NLP keyword risk signal
  └── Flood Alerts         →  weather hazard signal
          │
          
  Feature Engineering (per LinkID × timestep)
          │
          ▼
  Autoencoder  →  Reconstruction Error  →  Anomaly Score
          │
          ▼
  Domain Multipliers  →  Risk Index [0–100]
```

## Notebook Sections
1. [Setup & API Config](#1-setup)
2. [API Data Fetchers](#2-fetchers)
3. [Feature Engineering](#3-features)
4. [Data Collection Loop](#4-collection)
5. [LSTM Autoencoder Model](#5-model)
6. [Training](#6-training)
7. [Risk Index Inference](#7-inference)


## 1. Setup & API Configuration <a id='1-setup'></a>

In [ ]:
# ── Install / import dependencies ──────────────────────────────────────
# All standard; only torch may need installing in fresh envs.
# !pip install torch requests pandas numpy scikit-learn matplotlib folium tqdm

import os, json, time, math, warnings, re
from datetime import datetime, timedelta
from collections import defaultdict, deque
from pathlib import Path

import requests
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


In [ ]:
# ── LTA DataMall API Configuration ──────────────────────────────────────

API_KEY = '' #hidden for security

BASE_URL = 'https://datamall2.mytransport.sg/ltaodataservice'

HEADERS = {
    'AccountKey': API_KEY,
    'accept': 'application/json'
}

# ── Collection parameters ────────────────────────────────────────────────
POLL_INTERVAL_SEC = 120         # poll every 2 minutes (matches fastest API refresh)
SEQUENCE_LENGTH = 12            # 12 timesteps × 2 min = 24-min look-back window
MIN_SAMPLES_TRAIN = 500        # minimum feature vectors before first training run
RETRAIN_EVERY = 300            # retrain every N new samples
DATA_DIR = Path('lta_data')
DATA_DIR.mkdir(exist_ok=True)

# ── Road-category baseline expected speeds (SpeedBand 1–8) ───────────────
# Used to compute deviation: how abnormal is the current speed for this road type?
# Based on LTA documentation: 1=Expressways, 2=Major Arterial, ...
ROAD_CAT_EXPECTED_BAND = {
    1: 7.0,   # Expressways: expect SpeedBand ~7 (60-69 km/h) normally
    2: 5.5,   # Major Arterial Roads
    3: 4.5,   # Arterial Roads
    4: 3.5,   # Minor Arterial Roads
    5: 3.0,   # Small Roads
    6: 3.0,   # Slip Roads
    8: 5.0,   # Short Tunnels
}

# ── Incident type risk weights ───────────────────────────────────────────
# Higher weight = more dangerous incident type
INCIDENT_RISK_WEIGHTS = {
    'Accident':           1.0,
    'Vehicle breakdown':  0.6,
    'Roadwork':           0.4,
    'Weather':            0.7,
    'Obstacle':           0.5,
    'Road Block':         0.5,
    'Heavy Traffic':      0.3,
    'Miscellaneous':      0.2,
    'Diversion':          0.3,
    'Unattended Vehicle': 0.5,
    'Fire':               0.9,
    'Plant Failure':      0.4,
    'Reverse Flow':       0.8,
}

# ── Domain multipliers applied on top of anomaly score ───────────────────
DOMAIN_MULTIPLIERS = {
    'active_accident':       2.0,
    'flood_extreme':         1.8,
    'flood_severe':          1.5,
    'flood_moderate':        1.3,
    'flood_minor':           1.1,
    'faulty_light_blackout': 1.6,
    'faulty_light_flashing': 1.3,
    'late_night':            1.3,   # 0200–0500 hrs
    'peak_hour':             1.1,   # 0730–0930 or 1730–1930
    'active_roadworks':      1.2,
    'emas_breakdown':        1.4,
    'emas_rain':             1.2,
}

print('Configuration loaded.')
print(f'API Base URL : {BASE_URL}')
print(f'Poll interval: {POLL_INTERVAL_SEC}s | Sequence length: {SEQUENCE_LENGTH} steps')


## 2. API Data Fetchers <a id='2-fetchers'></a>

Each fetcher maps directly to one LTA DataMall API endpoint.  
Paginated endpoints use `$skip` to retrieve all records (max 500/call per the docs).


In [ ]:
# ── Generic paginated fetch ──────────────────────────────────────────────
def fetch_paginated(endpoint: str, params: dict = None) -> list:
    """
    Fetches all records from a paginated LTA API endpoint.
    Per the API docs, responses are capped at 500 records per call;
    use $skip to retrieve subsequent pages.
    """
    results = []
    skip = 0
    params = params or {}
    while True:
        p = {**params, '$skip': skip}
        try:
            r = requests.get(f'{BASE_URL}/{endpoint}', headers=HEADERS, params=p, timeout=10)
            r.raise_for_status()
            batch = r.json().get('value', [])
        except Exception as e:
            print(f'[WARN] {endpoint} skip={skip} failed: {e}')
            break
        if not batch:
            break
        results.extend(batch)
        if len(batch) < 500:   # last page
            break
        skip += 500
    return results


def fetch_single(endpoint: str, params: dict = None) -> dict:
    """Fetches a single-response endpoint (no pagination)."""
    try:
        r = requests.get(f'{BASE_URL}/{endpoint}', headers=HEADERS,
                         params=params or {}, timeout=10)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'[WARN] {endpoint} failed: {e}')
        return {}


# ── 2.1  Traffic Speed Bands (API 2.19) — core feature source ────────────
# URL: /v4/TrafficSpeedBands  | Refresh: 5 minutes
# Returns: LinkID, RoadName, RoadCategory (1-8), SpeedBand (1-8),
#          MinimumSpeed, MaximumSpeed, StartLon, StartLat, EndLon, EndLat
def fetch_speed_bands() -> pd.DataFrame:
    records = fetch_paginated('v4/TrafficSpeedBands')
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    # Cast numerics — API returns strings for some fields
    for col in ['SpeedBand', 'MinimumSpeed', 'MaximumSpeed',
                'StartLon', 'StartLat', 'EndLon', 'EndLat', 'RoadCategory']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['timestamp'] = datetime.now()
    return df


# ── 2.2  Traffic Incidents (API 2.18) — weak supervision signal ───────────
# URL: /TrafficIncidents  | Refresh: 2 minutes
# Returns: Type, Latitude, Longitude, Message
def fetch_incidents() -> pd.DataFrame:
    records = fetch_paginated('TrafficIncidents')
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    for col in ['Latitude', 'Longitude']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['timestamp'] = datetime.now()
    return df


# ── 2.3  Faulty Traffic Lights (API 2.14) ────────────────────────────────
# URL: /FaultyTrafficLights  | Refresh: 2 minutes
# Type: 4 = Blackout, 13 = Flashing Yellow
# Note: NodeID is a traffic light node; we use coordinates from Speed Bands
#       to proxy geospatial join (no lat/lon in this API — use message parsing)
def fetch_faulty_lights() -> pd.DataFrame:
    records = fetch_paginated('FaultyTrafficLights')
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    if 'Type' in df.columns:
        df['Type'] = pd.to_numeric(df['Type'], errors='coerce')
    df['timestamp'] = datetime.now()
    return df


# ── 2.4  VMS / EMAS (API 2.20) ───────────────────────────────────────────
# URL: /VMS  | Refresh: 2 minutes
# Returns: EquipmentID, Latitude, Longitude, Message
def fetch_vms() -> pd.DataFrame:
    records = fetch_paginated('VMS')
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    for col in ['Latitude', 'Longitude']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['timestamp'] = datetime.now()
    return df


# ── 2.5  Flood Alerts (API 2.30) ─────────────────────────────────────────
# URL: /PubFloodAlerts  | Refresh: 3 minutes
# Returns: alertId, dateTime, severity (Extreme/Severe/Moderate/Minor),
#          description, circle (lat,lon radius_km), areaDesc
def fetch_flood_alerts() -> pd.DataFrame:
    records = fetch_paginated('PubFloodAlerts')
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    # Parse circle field: 'lat,lon radius'
    if 'circle' in df.columns:
        def parse_circle(s):
            try:
                parts = str(s).split()
                latlon = parts[0].split(',')
                return float(latlon[0]), float(latlon[1]), float(parts[1])
            except:
                return None, None, None
        parsed = df['circle'].apply(parse_circle)
        df['flood_lat']    = parsed.apply(lambda x: x[0])
        df['flood_lon']    = parsed.apply(lambda x: x[1])
        df['flood_radius'] = parsed.apply(lambda x: x[2])
    df['timestamp'] = datetime.now()
    return df


print('API fetchers defined.')


In [ ]:
# ──––––––––––––––– test each fetcher ───────────–––––––––

def test_fetchers():
    print('Testing API fetchers...')
    results = {}

    df_speed = fetch_speed_bands()
    results['speed_bands'] = len(df_speed)
    print(f'  Speed Bands:      {len(df_speed):>5} records')
    if not df_speed.empty:
        print(f'    Columns: {list(df_speed.columns)}')
        print(f'    Sample:\n{df_speed.head(2).to_string()}')

    df_inc = fetch_incidents()
    results['incidents'] = len(df_inc)
    print(f'  Incidents:        {len(df_inc):>5} records')

    df_fl = fetch_faulty_lights()
    results['faulty_lights'] = len(df_fl)
    print(f'  Faulty Lights:    {len(df_fl):>5} records')

    df_vms = fetch_vms()
    results['vms'] = len(df_vms)
    print(f'  VMS/EMAS:         {len(df_vms):>5} records')

    df_flood = fetch_flood_alerts()
    results['floods'] = len(df_flood)
    print(f'  Flood Alerts:     {len(df_flood):>5} records')

    return results

#test_results = test_fetchers()
print('Run test_fetchers() to verify')


## 3. Feature Engineering <a id='3-features'></a>

For each road segment (`LinkID`) at each timestep we build a multidimensional feature vector:

| Feature Group | Features |
|---|---|
| Speed | `speed_band`, `min_speed`, `max_speed` |
| Speed anomaly | `speed_deviation`, `speed_drop_rate` |
| Incidents | `incident_score`, `incident_count`, `nearest_incident_dist` |
| Incident types | 4 binary flags - accident/breakdown/weather/roadwork |
| Traffic lights | `faulty_light_score`, `blackout_nearby`, `flashing_nearby` |
| VMS / EMAS | `emas_breakdown_score`, `emas_rain_score` |
| Flood | `flood_severity_score`, `flood_nearby` | 
| Road metadata | `road_category` (one-hot, 7 dim) | 
| Time | `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos` |
| Derived | `is_peak_hour`, `is_late_night` |


In [ ]:
# ── Haversine distance (km) between two lat/lon points ───────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


def segment_midpoint(row):
    """Mid-point of a road segment from its start/end coordinates."""
    lat = (row.get('StartLat', 0) + row.get('EndLat', 0)) / 2
    lon = (row.get('StartLon', 0) + row.get('EndLon', 0)) / 2
    return lat, lon


# ── Incident scoring for a single segment ────────────────────────────────
PROXIMITY_THRESHOLD_KM = 1.0   # incidents within 1 km affect the segment

def compute_incident_features(seg_lat, seg_lon, df_incidents):
    """
    Returns a dict of incident-derived features for a road segment.
    All incidents within PROXIMITY_THRESHOLD_KM are considered.
    """
    feats = {
        'incident_score': 0.0,
        'incident_count': 0,
        'nearest_incident_dist': PROXIMITY_THRESHOLD_KM,  # max if no incident nearby
        'flag_accident': 0,
        'flag_breakdown': 0,
        'flag_weather': 0,
        'flag_roadwork': 0,
    }
    if df_incidents is None or df_incidents.empty:
        return feats

    nearby = []
    for _, inc in df_incidents.iterrows():
        try:
            dist = haversine(seg_lat, seg_lon,
                             float(inc['Latitude']), float(inc['Longitude']))
        except:
            continue
        if dist <= PROXIMITY_THRESHOLD_KM:
            nearby.append({'dist': dist, 'type': inc.get('Type', 'Miscellaneous')})

    if nearby:
        feats['incident_count'] = len(nearby)
        feats['nearest_incident_dist'] = min(n['dist'] for n in nearby)
        # Aggregate weighted score: weight by type × inverse-distance
        for n in nearby:
            w = INCIDENT_RISK_WEIGHTS.get(n['type'], 0.2)
            inv_dist = 1.0 / max(n['dist'], 0.05)
            feats['incident_score'] += w * inv_dist
            t = n['type'].lower()
            if 'accident' in t: feats['flag_accident'] = 1
            if 'breakdown' in t: feats['flag_breakdown'] = 1
            if 'weather' in t: feats['flag_weather'] = 1
            if 'roadwork' in t or 'road work' in t: feats['flag_roadwork'] = 1
        feats['incident_score'] = min(feats['incident_score'], 20.0)  # cap

    return feats


# ── Faulty traffic light scoring ─────────────────────────────────────────
# Note: FaultyTrafficLights API has no lat/lon — we use NodeID prefix heuristic
# and count of faults as a global-level signal, weighted by type severity.
# Type 4 = Blackout, Type 13 = Flashing Yellow
def compute_light_features(df_lights):
    """
    Returns global faulty-light signal (counts and type flags).
    Since the API provides no coordinates, we use count-based signals
    that apply network-wide, modulated by proximity to road category.
    """
    feats = {
        'faulty_light_score': 0.0,
        'blackout_count':     0,
        'flashing_count':     0,
    }
    if df_lights is None or df_lights.empty:
        return feats

    blackouts = df_lights[df_lights['Type'] == 4]
    flashings = df_lights[df_lights['Type'] == 13]
    feats['blackout_count'] = len(blackouts)
    feats['flashing_count'] = len(flashings)
    feats['faulty_light_score'] = len(blackouts) * 1.5 + len(flashings) * 0.8
    return feats


# ── VMS / EMAS keyword scoring ────────────────────────────────────────────
#based on LTA API
EMAS_BREAKDOWN_KEYWORDS = ['breakdown', 'brkdown', 'veh break', 'stalled']
EMAS_RAIN_KEYWORDS      = ['rain', 'wet', 'flood', 'slippery']

def compute_vms_features(seg_lat, seg_lon, df_vms):
    """
    Parses EMAS message text for risk keywords.
    VMS signboards within 2 km of the segment are considered.
    """
    feats = {'emas_breakdown_score': 0.0, 'emas_rain_score': 0.0}
    if df_vms is None or df_vms.empty:
        return feats

    EMAS_RADIUS_KM = 2.0
    for _, row in df_vms.iterrows():
        try:
            dist = haversine(seg_lat, seg_lon,
                             float(row['Latitude']), float(row['Longitude']))
        except:
            continue
        if dist > EMAS_RADIUS_KM:
            continue
        msg = str(row.get('Message', '')).lower()
        weight = 1.0 / max(dist, 0.1)
        if any(kw in msg for kw in EMAS_BREAKDOWN_KEYWORDS):
            feats['emas_breakdown_score'] += weight
        if any(kw in msg for kw in EMAS_RAIN_KEYWORDS):
            feats['emas_rain_score'] += weight

    feats['emas_breakdown_score'] = min(feats['emas_breakdown_score'], 10.0)
    feats['emas_rain_score']      = min(feats['emas_rain_score'], 10.0)
    return feats


# ── Flood alert scoring ───────────────────────────────────────────────────
SEVERITY_SCORE = {'Extreme': 4, 'Severe': 3, 'Moderate': 2, 'Minor': 1}

def compute_flood_features(seg_lat, seg_lon, df_flood):
    """
    Returns flood severity score for the segment.
    The 'circle' field gives broadcast radius (not flood extent per the API docs);
    we use it as the proximity threshold.
    """
    feats = {'flood_severity_score': 0.0, 'flood_nearby': 0}
    if df_flood is None or df_flood.empty:
        return feats

    for _, row in df_flood.iterrows():
        try:
            flat  = float(row['flood_lat'])
            flon  = float(row['flood_lon'])
            frad  = float(row.get('flood_radius', 0.5))
            dist  = haversine(seg_lat, seg_lon, flat, flon)
        except:
            continue
        if dist <= max(frad, 0.5):  # at least 500m radius
            sev   = SEVERITY_SCORE.get(str(row.get('severity', '')), 1)
            feats['flood_severity_score'] = max(feats['flood_severity_score'], sev)
            feats['flood_nearby'] = 1
    return feats


# ── Cyclical time encoding ────────────────────────────────────────────────
def cyclical_encode(value, max_value):
    """Encode a cyclical value (e.g., hour 0-23) as (sin, cos) pair."""
    angle = 2 * math.pi * value / max_value
    return math.sin(angle), math.cos(angle)


def time_features(dt: datetime):
    h_sin, h_cos = cyclical_encode(dt.hour, 24)
    d_sin, d_cos = cyclical_encode(dt.weekday(), 7)
    # Peak hour: 0730–0930 or 1730–1930
    is_peak = int((7.5 <= dt.hour + dt.minute/60 <= 9.5) or
                  (17.5 <= dt.hour + dt.minute/60 <= 19.5))
    # Late night: 0200–0500
    is_late_night = int(2 <= dt.hour <= 5)
    return {
        'hour_sin': h_sin,
        'hour_cos': h_cos,
        'dow_sin': d_sin,
        'dow_cos': d_cos,
        'is_peak_hour': is_peak,
        'is_late_night': is_late_night,
    }


print('Feature engineering functions defined.')


In [ ]:
# ── Master feature builder: one row per (LinkID, timestamp) ──────────────

ROAD_CAT_ONEHOT_COLS = [f'road_cat_{c}' for c in [1,2,3,4,5,6,8]]

FEATURE_COLS = [
    # Speed (3)
    'speed_band', 'min_speed', 'max_speed',
    # Speed anomaly (2)
    'speed_deviation', 'speed_drop_rate',
    # Incidents (7)
    'incident_score', 'incident_count', 'nearest_incident_dist',
    'flag_accident', 'flag_breakdown', 'flag_weather', 'flag_roadwork',
    # Traffic lights (3)
    'faulty_light_score', 'blackout_count', 'flashing_count',
    # VMS (2)
    'emas_breakdown_score', 'emas_rain_score',
    # Flood (2)
    'flood_severity_score', 'flood_nearby',
    # Road category one-hot (7)
    *ROAD_CAT_ONEHOT_COLS,
    # Time (6)
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'is_peak_hour', 'is_late_night',
]
FEATURE_DIM = len(FEATURE_COLS)
print(f'Feature vector dimension: {FEATURE_DIM}')
print('Features:', FEATURE_COLS)


def build_feature_row(seg: dict, prev_speed_band: float,
                      df_incidents, df_lights, df_vms, df_flood,
                      dt: datetime) -> dict:
    """
    Build a single feature row for a road segment at a given time.

    Args:
        seg:            dict of one row from the speed bands DataFrame
        prev_speed_band: SpeedBand value at previous timestep (for speed_drop_rate)
        df_incidents:   current incidents DataFrame
        df_lights:      current faulty lights DataFrame
        df_vms:         current VMS DataFrame
        df_flood:       current flood alerts DataFrame
        dt:             current datetime

    Returns:
        dict mapping FEATURE_COLS names to float values
    """
    seg_lat, seg_lon = segment_midpoint(seg)

    road_cat = int(seg.get('RoadCategory', 5))
    speed_band = float(seg.get('SpeedBand', 4))
    min_speed  = float(seg.get('MinimumSpeed', 0))
    max_speed  = float(seg.get('MaximumSpeed', 0))

    # Speed deviation from expected band for this road category
    expected_band   = ROAD_CAT_EXPECTED_BAND.get(road_cat, 4.0)
    speed_deviation = expected_band - speed_band  # positive = slower than expected

    # Speed drop rate from previous observation (positive = slowing down)
    speed_drop_rate = float(prev_speed_band - speed_band) if prev_speed_band is not None else 0.0

    # Road category one-hot
    road_cat_oh = {f'road_cat_{c}': int(road_cat == c) for c in [1,2,3,4,5,6,8]}

    row = {
        'link_id':   seg.get('LinkID', 'UNKNOWN'),
        'timestamp': dt,
        'road_name': seg.get('RoadName', ''),
        'seg_lat':   seg_lat,
        'seg_lon':   seg_lon,
        # Speed features
        'speed_band':      speed_band,
        'min_speed':       min_speed,
        'max_speed':       max_speed,
        'speed_deviation': speed_deviation,
        'speed_drop_rate': speed_drop_rate,
        # Road cat one-hot
        **road_cat_oh,
        # Time features
        **time_features(dt),
    }

    # Incident features
    row.update(compute_incident_features(seg_lat, seg_lon, df_incidents))

    # Faulty light features (global signal — same for all segments this tick)
    row.update(compute_light_features(df_lights))

    # VMS features
    row.update(compute_vms_features(seg_lat, seg_lon, df_vms))

    # Flood features
    row.update(compute_flood_features(seg_lat, seg_lon, df_flood))

    return row


print('build_feature_row() defined.')


## 4. Data Collection Loop <a id='4-collection'></a>

This section collects time-series feature data and persists it to disk.  
Alternatively, load previously collected data from `lta_data/features.parquet`


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# ── Per-segment speed history ─────────────────────────────────────────────
prev_speed_bands: dict = {}
feature_buffer:   list = []

# Precompute lookup structures once at module load
_ROAD_CAT_EXPECTED = pd.Series(ROAD_CAT_EXPECTED_BAND)
_INC_WEIGHT_MAP = pd.Series(INCIDENT_RISK_WEIGHTS)
_BREAKDOWN_RE = re.compile(r'breakdown|brkdown|veh break|stalled', re.I)
_RAIN_RE = re.compile(r'rain|wet|flood|slippery',             re.I)

# ── Vectorised feature builders ───────────────────────────────────────────
# Each function takes the full speed-bands DataFrame + one auxiliary DataFrame
# and returns a columns-only DataFrame indexed by LinkID position.
# No Python loops over segments — all ops are numpy/pandas broadcasting.

def _feat_speed(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out['seg_lat'] = (df['StartLat'].fillna(0) + df['EndLat'].fillna(0)) / 2
    out['seg_lon'] = (df['StartLon'].fillna(0) + df['EndLon'].fillna(0)) / 2
    out['speed_band'] = df['SpeedBand']
    out['min_speed'] = df['MinimumSpeed']
    out['max_speed'] = df['MaximumSpeed']
    expected = df['RoadCategory'].map(_ROAD_CAT_EXPECTED).fillna(4.0)
    out['speed_deviation']= expected - df['SpeedBand']
    prev = df['LinkID'].map(prev_speed_bands)
    out['speed_drop_rate']= (prev - df['SpeedBand']).fillna(0.0)
    for c in [1, 2, 3, 4, 5, 6, 8]:
        out[f'road_cat_{c}'] = (df['RoadCategory'] == c).astype(np.float32)
    return out


def _vectorised_haversine(lat1, lon1, lat2, lon2):
    """
    Broadcast haversine for (N,) vs (M,) arrays → returns (N, M) distance matrix in km.
    lat1/lon1 shape (N,), lat2/lon2 shape (M,)  →  output shape (N, M)
    """
    R = 6371.0
    lat1 = np.radians(lat1[:, None]);  lat2 = np.radians(lat2[None, :])
    dlat = lat2 - lat1
    dlon = np.radians(lon2[None, :] - lon1[:, None])
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(np.maximum(1 - a, 0)))


def _feat_incidents(df: pd.DataFrame, df_inc: pd.DataFrame,
                    radius_km: float = 1.0) -> pd.DataFrame:
    cols = ['incident_score', 'incident_count', 'nearest_incident_dist',
            'flag_accident', 'flag_breakdown', 'flag_weather', 'flag_roadwork']
    empty = pd.DataFrame(
        {c: np.zeros(len(df)) for c in cols}
        | {'nearest_incident_dist': np.full(len(df), radius_km)},
        index=df.index
    )
    if df_inc is None or df_inc.empty:
        return empty

    dist = _vectorised_haversine(          # (N_segs, N_inc)
        df['seg_lat'].values, df['seg_lon'].values,
        df_inc['Latitude'].values.astype(float),
        df_inc['Longitude'].values.astype(float),
    )
    mask = dist <= radius_km            # (N_segs, N_inc) bool
    weights = np.array([INCIDENT_RISK_WEIGHTS.get(t, 0.2)
                        for t in df_inc['Type'].fillna('Miscellaneous')])
    inv_d   = np.where(mask, 1.0 / np.maximum(dist, 0.05), 0.0)

    out = pd.DataFrame(index=df.index)
    out['incident_count'] = mask.sum(axis=1)
    nearest = np.where(mask, dist, np.inf).min(axis=1)
    out['nearest_incident_dist'] = np.where(nearest == np.inf, radius_km, nearest)
    out['incident_score'] = np.minimum((inv_d * weights).sum(axis=1), 20.0)

    types = df_inc['Type'].fillna('').str.lower().values
    for flag, kw in [('flag_accident', ['accident']),
                     ('flag_breakdown', ['breakdown']),
                     ('flag_weather', ['weather']),
                     ('flag_roadwork', ['roadwork', 'road work'])]:
        match = np.array([any(k in t for k in kw) for t in types])  # (N_inc,)
        out[flag] = (mask & match[None, :]).any(axis=1).astype(int)

    return out


def _feat_vms(df: pd.DataFrame, df_vms: pd.DataFrame,
              radius_km: float = 2.0) -> pd.DataFrame:
    empty = pd.DataFrame({'emas_breakdown_score': 0.0, 'emas_rain_score': 0.0},
                         index=df.index)
    if df_vms is None or df_vms.empty:
        return empty
 
    dist = _vectorised_haversine(
        df['seg_lat'].values, df['seg_lon'].values,
        df_vms['Latitude'].values.astype(float),
        df_vms['Longitude'].values.astype(float),
    )
    mask = dist <= radius_km
    inv_d = np.where(mask, 1.0 / np.maximum(dist, 0.1), 0.0)
    msgs = df_vms['Message'].fillna('').values
    is_brkd = np.array([bool(_BREAKDOWN_RE.search(m)) for m in msgs])
    is_rain = np.array([bool(_RAIN_RE.search(m))      for m in msgs])

    out = pd.DataFrame(index=df.index)
    out['emas_breakdown_score'] = np.minimum((inv_d * is_brkd).sum(axis=1), 10.0)
    out['emas_rain_score'] = np.minimum((inv_d * is_rain).sum(axis=1),  10.0)
    return out


def _feat_flood(df: pd.DataFrame, df_flood: pd.DataFrame) -> pd.DataFrame:
    empty = pd.DataFrame({'flood_severity_score': 0.0, 'flood_nearby': 0},
                         index=df.index)
    if df_flood is None or df_flood.empty:
        return empty
    valid = df_flood.dropna(subset=['flood_lat', 'flood_lon'])
    if valid.empty:
        return empty

    dist = _vectorised_haversine(
        df['seg_lat'].values, df['seg_lon'].values,
        valid['flood_lat'].values.astype(float),
        valid['flood_lon'].values.astype(float),
    )
    radii = np.maximum(valid.get('flood_radius', pd.Series(0.5)).fillna(0.5).values, 0.5)
    sev = valid['severity'].map(SEVERITY_SCORE).fillna(1).values.astype(float)
    in_zone = dist <= radii[None, :]

    out = pd.DataFrame(index=df.index)
    out['flood_severity_score'] = (in_zone * sev[None, :]).max(axis=1)
    out['flood_nearby']         = (out['flood_severity_score'] > 0).astype(int)
    return out


def _feat_lights(df: pd.DataFrame, df_lights: pd.DataFrame) -> pd.DataFrame:
    # No coordinates in this API — broadcast global counts to every segment
    n_black = int((df_lights['Type'] == 4).sum())  if df_lights is not None and not df_lights.empty else 0
    n_flash = int((df_lights['Type'] == 13).sum()) if df_lights is not None and not df_lights.empty else 0
    n = len(df)
    return pd.DataFrame({
        'faulty_light_score': np.full(n, n_black * 1.5 + n_flash * 0.8),
        'blackout_count': np.full(n, n_black, dtype=int),
        'flashing_count': np.full(n, n_flash, dtype=int),
    }, index=df.index)


def _feat_time(df: pd.DataFrame, dt: datetime) -> pd.DataFrame:
    tf = time_features(dt)                         # existing scalar helper
    return pd.DataFrame({k: np.full(len(df), v) for k, v in tf.items()},
                        index=df.index)


# ── Main snapshot ─────────────────────────────────────────────────────────

def collect_one_snapshot() -> int:
    global prev_speed_bands, feature_buffer
    now = datetime.now()
    t0 = time.perf_counter()

    # All 5 APIs fetched concurrently
    fetch_jobs = dict(speed=fetch_speed_bands, inc=fetch_incidents,
                      lights=fetch_faulty_lights, vms=fetch_vms, flood=fetch_flood_alerts)
    fetched = {}
    with ThreadPoolExecutor(max_workers=5) as pool:
        futs = {pool.submit(fn): name for name, fn in fetch_jobs.items()}
        for fut in as_completed(futs):
            fetched[futs[fut]] = fut.result()

    df_speed = fetched['speed']
    if df_speed.empty:
        print(f'[{now:%H:%M:%S}] Speed bands empty — skipping.')
        return 0
    t_fetch = time.perf_counter() - t0

    # All feature blocks built — speed first (provides seg_lat/lon for the rest)
    df_speed = df_speed.reset_index(drop=True)
    blocks = [_feat_speed(df_speed)]                          # must be first
    df_speed = df_speed.copy()
    df_speed[['seg_lat','seg_lon']] = blocks[0][['seg_lat','seg_lon']]

    # Remaining blocks are independent — build in parallel
    with ThreadPoolExecutor(max_workers=4) as pool:
        futs2 = [
            pool.submit(_feat_incidents, df_speed, fetched['inc']),
            pool.submit(_feat_vms, df_speed, fetched['vms']),
            pool.submit(_feat_flood, df_speed, fetched['flood']),
            pool.submit(_feat_lights, df_speed, fetched['lights']),
        ]
        for fut in futs2:
            blocks.append(fut.result())
    blocks.append(_feat_time(df_speed, now))

    # Single concat — one allocation, no per-row appending
    df_feat = pd.concat(blocks, axis=1)
    df_feat['LinkID'] = df_speed['LinkID'].values
    df_feat['road_name'] = df_speed['RoadName'].values
    df_feat['timestamp'] = now

    # Ensure all expected columns present
    for c in FEATURE_COLS:
        if c not in df_feat.columns:
            df_feat[c] = 0.0

    # Update prev_speed_bands in one dict update
    prev_speed_bands.update(
        zip(df_speed['LinkID'].values, df_speed['SpeedBand'].values.astype(float))
    )

    feature_buffer.extend(df_feat.to_dict('records'))

    t_total = time.perf_counter() - t0
    print(f'[{now:%H:%M:%S}] {len(df_feat)} segments | '
          f'fetch={t_fetch:.2f}s  build={t_total-t_fetch:.3f}s  total={t_total:.2f}s')
    return len(df_feat)


def save_features(path: Path = DATA_DIR / 'features.parquet'):
    if not feature_buffer:
        print('Buffer is empty, nothing to save.')
        return
    df = pd.DataFrame(feature_buffer)
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    df[FEATURE_COLS] = df[FEATURE_COLS].astype(np.float32)   # halves file size
    df.to_parquet(path, index=False, compression='snappy')
    print(f'Saved {len(df):,} rows → {path}  ({path.stat().st_size/1024:.0f} KB)')


def load_features(path: Path = DATA_DIR / 'features.parquet') -> pd.DataFrame:
    if not path.exists():
        print(f'No existing data at {path}')
        return pd.DataFrame()
    df = pd.read_parquet(path)
    print(f'Loaded {len(df):,} rows from {path}')
    return df


print('Data collection functions defined (vectorised).')
print('Expected per-snapshot: ~2-4s total (dominated by network)')

In [ ]:
# ── OPTION A: Run live collection ─────────────────────────────────────────
# Runs N snapshots (N × 2 min ≈ collection duration).
# For training you need at least MIN_SAMPLES_TRAIN total feature rows.
# With ~1000 road segments, 1 snapshot = ~1000 rows → 1 snapshot is enough for
# the autoencoder; but for the time-series sequences you need SEQUENCE_LENGTH
# consecutive snapshots per segment.

def run_collection(n_snapshots: int = 20, interval_sec: int = POLL_INTERVAL_SEC,
                   save_every: int = 5):
    """
    Collect `n_snapshots` API polls spaced `interval_sec` seconds apart.
    Saves to disk every `save_every` snapshots.

    n_snapshots=20 → 40 minutes of data at 2-min intervals.
    n_snapshots=60 → 2 hours of data (recommended minimum for training).
    """
    print(f'Starting collection: {n_snapshots} snapshots × {interval_sec}s = '
          f'{n_snapshots * interval_sec / 60:.1f} minutes')

    for i in range(n_snapshots):
        collect_one_snapshot()
        if (i + 1) % save_every == 0:
            save_features()
        if i < n_snapshots - 1:
            time.sleep(interval_sec)

    save_features()  # final save
    print(f'Collection complete. Total buffer size: {len(feature_buffer)} rows.')


# Uncomment ONE of the following:

# ► Live collection (requires valid API key + time):
run_collection(n_snapshots=60)   # ~2 hours

print('run_collection() defined — uncomment above to start collecting data.')


In [ ]:
# ── OPTION B: Generate synthetic data for offline development ─────────────
# This generates realistic synthetic time-series data mimicking the API output,
# letting you develop and test the model without waiting for real data collection.

def generate_synthetic_data(
        n_segments: int = 200,
        n_timesteps: int = 100,
        anomaly_frac: float = 0.05,
        seed: int = SEED
) -> pd.DataFrame:
    """
    Synthetic dataset that mimics real LTA feature distributions.
    Introduces deliberate anomalies (sudden speed drops + active incidents)
    so we can verify the autoencoder detects them.

    Returns a DataFrame with columns = metadata cols + FEATURE_COLS.
    """
    rng = np.random.default_rng(seed)
    rows = []
    base_time = datetime(2026, 3, 1, 7, 0, 0)  # Monday 07:00

    # Assign fixed road categories to segments (skewed toward real distribution)
    road_cats = rng.choice([1,2,3,4,5,6], size=n_segments,
                           p=[0.10, 0.20, 0.25, 0.20, 0.15, 0.10])
    # Fixed lat/lon within Singapore bounding box
    seg_lats = rng.uniform(1.25, 1.45, size=n_segments)
    seg_lons = rng.uniform(103.65, 104.0, size=n_segments)

    for t in range(n_timesteps):
        dt = base_time + timedelta(minutes=2 * t)
        tf = time_features(dt)

        # Global signals: vary slightly over time
        n_blackouts = int(rng.poisson(0.5))   # avg 0.5 blackouts active
        n_flashings = int(rng.poisson(1.0))
        faulty_light_score = n_blackouts * 1.5 + n_flashings * 0.8
        flood_sev_global   = 0.0  # usually 0

        # Randomly inject flood event for a window
        if 40 <= t < 50:
            flood_sev_global = 2.0  # Moderate flood

        for s in range(n_segments):
            rc = int(road_cats[s])
            expected_band = ROAD_CAT_EXPECTED_BAND.get(rc, 4.0)

            # Normal speed: Gaussian around expected band, clipped [1,8]
            speed_band = float(np.clip(rng.normal(expected_band, 0.8), 1, 8))

            # Inject anomalies for ~5% of (segment, timestep) pairs
            is_anomaly = rng.random() < anomaly_frac
            flag_acc = 0
            incident_score = 0.0
            if is_anomaly:
                speed_band = max(1.0, speed_band - rng.uniform(2, 4))  # sudden drop
                flag_acc   = 1
                incident_score = rng.uniform(2, 10)

            min_speed = max(0, (speed_band - 1) * 10)
            max_speed = speed_band * 10
            speed_deviation = expected_band - speed_band
            speed_drop_rate = rng.uniform(-0.3, 0.5) + (2.0 if is_anomaly else 0)

            road_cat_oh = {f'road_cat_{c}': int(rc == c) for c in [1,2,3,4,5,6,8]}

            row = {
                'link_id':    f'SEG_{s:04d}',
                'timestamp':  dt,
                'road_name':  f'Synthetic Road {s}',
                'seg_lat':    float(seg_lats[s]),
                'seg_lon':    float(seg_lons[s]),
                'is_anomaly': is_anomaly,  # ground truth label (only for evaluation)
                # Speed
                'speed_band':      speed_band,
                'min_speed':       min_speed,
                'max_speed':       max_speed,
                'speed_deviation': speed_deviation,
                'speed_drop_rate': speed_drop_rate,
                # Incidents
                'incident_score':        incident_score,
                'incident_count':        int(flag_acc),
                'nearest_incident_dist': rng.uniform(0.1, 1.0) if flag_acc else 1.0,
                'flag_accident':         flag_acc,
                'flag_breakdown':        int(rng.random() < 0.02),
                'flag_weather':          int(rng.random() < 0.01),
                'flag_roadwork':         int(rng.random() < 0.03),
                # Traffic lights
                'faulty_light_score': faulty_light_score,
                'blackout_count':     n_blackouts,
                'flashing_count':     n_flashings,
                # VMS
                'emas_breakdown_score': rng.uniform(0, 2) if is_anomaly else 0.0,
                'emas_rain_score':      rng.uniform(0, 1.5) if flood_sev_global > 0 else 0.0,
                # Flood
                'flood_severity_score': flood_sev_global,
                'flood_nearby':         int(flood_sev_global > 0),
                # Time
                **tf,
                # Road cat one-hot
                **road_cat_oh,
            }
            rows.append(row)

    df = pd.DataFrame(rows)
    print(f'Synthetic dataset: {len(df)} rows | {n_segments} segments × {n_timesteps} timesteps')
    print(f'Anomaly rate: {df["is_anomaly"].mean():.2%}')
    return df


# Generate synthetic data (use this if you don't have a real API key yet)
df_synthetic = generate_synthetic_data(n_segments=300, n_timesteps=120)
df_synthetic.head(3)


In [ ]:
# ── Select which dataset to use for training ─────────────────────────────
# Switch USING_REAL_DATA = True once you have collected real data.

USING_REAL_DATA = True

if USING_REAL_DATA:
    df_all = load_features()          # load from lta_data/features.parquet
    if df_all.empty:
        raise RuntimeError('No real data found. Run run_collection() first.')
else:
    # Use synthetic data — 'is_anomaly' column available for evaluation
    df_all = df_synthetic.copy()

print(f'Dataset shape: {df_all.shape}')
print(f'Unique segments: {df_all["LinkID"].nunique()}')
print(f'Time range: {df_all["timestamp"].min()} → {df_all["timestamp"].max()}')
print(f'Feature columns present: {[c for c in FEATURE_COLS if c in df_all.columns]}')
df_all[FEATURE_COLS].describe().round(3)


## 5. Autoencoder Model <a id='5-model'></a>

### How it works
- Trained **only on normal data** → learns to reconstruct normal traffic patterns well
- **High reconstruction error** at inference = abnormal conditions = elevated risk

```
Input sequence  [t-11, ..., t-1, t]  shape: (seq_len, feature_dim)
      │
  Autoencoder Model
      │
  MSE Loss = reconstruction_error
      │
  anomaly_score = (reconstruction_error - μ) / σ   [z-scored against baseline]
```


In [ ]:
# ── Dataset: build sliding-window sequences per segment ──────────────────

class RoadSegmentDataset(Dataset):
    """
    Creates sliding-window sequences of shape (SEQUENCE_LENGTH, FEATURE_DIM)
    from the time-series feature DataFrame.

    Only includes sequences where all SEQUENCE_LENGTH steps are consecutive
    (same link_id, no time gap > 3× poll interval).

    'normal_only': if True, exclude rows marked as anomalies (for training).
    """

    def __init__(self, df: pd.DataFrame, feature_cols: list,
                 seq_len: int = SEQUENCE_LENGTH,
                 scaler: StandardScaler = None,
                 fit_scaler: bool = True,
                 normal_only: bool = False,
                 max_gap_min: float = 6.0):
        self.seq_len      = seq_len
        self.feature_cols = feature_cols
        self.sequences    = []   # list of np arrays (seq_len, feature_dim)
        self.meta         = []   # (link_id, end_timestamp) for each sequence
        self.labels       = []   # is_anomaly for last step (if available)

        df = df.sort_values(['LinkID', 'timestamp']).copy()

        # Filter to normal data only (for training)
        if normal_only and 'is_anomaly' in df.columns:
            df = df[~df['is_anomaly']]

        # Ensure feature columns exist
        for c in feature_cols:
            if c not in df.columns:
                df[c] = 0.0

        X_raw = df[feature_cols].values.astype(np.float32)

        # Fit or apply scaler
        if scaler is None:
            self.scaler = StandardScaler()
        else:
            self.scaler = scaler
        if fit_scaler:
            X_scaled = self.scaler.fit_transform(X_raw)
        else:
            X_scaled = self.scaler.transform(X_raw)
        df['_scaled_idx'] = np.arange(len(df))

        max_gap = timedelta(minutes=max_gap_min)

        for link_id, grp in df.groupby('LinkID', sort=False):
            grp = grp.sort_values('timestamp').reset_index(drop=True)
            idxs = grp['_scaled_idx'].values
            times = pd.to_datetime(grp['timestamp'].values)
            anom  = grp['is_anomaly'].values if 'is_anomaly' in grp.columns else \
                    np.zeros(len(grp), dtype=bool)

            for i in range(len(grp) - seq_len + 1):
                # Check all consecutive within max_gap
                window_times = times[i:i+seq_len]
                gaps = [(window_times[j+1] - window_times[j]) for j in range(seq_len-1)]
                if any(g > max_gap for g in gaps):
                    continue
                seq_idxs = idxs[i:i+seq_len]
                self.sequences.append(X_scaled[seq_idxs])    # (seq_len, feat_dim)
                self.meta.append((link_id, times[i+seq_len-1]))
                self.labels.append(bool(anom[i+seq_len-1]))

        print(f'Dataset built: {len(self.sequences)} sequences '
              f'({seq_len} steps × {len(feature_cols)} features)')
        if len(self.sequences) == 0:
            print('  ⚠ No valid sequences found. Ensure you have >= seq_len consecutive '
                  'timestamps per segment.')

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.float32)


print('RoadSegmentDataset defined.')


In [ ]:
# ── Feedforward Autoencoder ───────────────────────────────────────────────
# Replaces the LSTM autoencoder. Each row is scored independently —
# no sequence dimension, no gap checking, no sliding window redundancy.
# The temporal signal is captured by the engineered speed_drop_rate feature.

class FeedforwardAutoencoder(nn.Module):
    def __init__(self, feature_dim: int = FEATURE_DIM,
                 hidden_dims: list = [32, 16, 8],
                 dropout: float = 0.2):
        super().__init__()

        enc, in_dim = [], feature_dim
        for h in hidden_dims:
            enc += [nn.Linear(in_dim, h), nn.ReLU(),
                    nn.BatchNorm1d(h), nn.Dropout(dropout)]
            in_dim = h
        self.encoder = nn.Sequential(*enc)

        dec = []
        for h in reversed(hidden_dims[:-1]):
            dec += [nn.Linear(in_dim, h), nn.ReLU(),
                    nn.BatchNorm1d(h), nn.Dropout(dropout)]
            in_dim = h
        dec.append(nn.Linear(in_dim, feature_dim))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def reconstruction_error(self, x):
        self.eval()
        with torch.no_grad():
            return ((x - self(x)) ** 2).mean(dim=1)   # (batch,)


model = FeedforwardAutoencoder(feature_dim=FEATURE_DIM, hidden_dims = [64,32], dropout=0.1).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'FeedforwardAutoencoder | params: {n_params:,} | device: {DEVICE}')
print(model)


## 6. Training <a id='6-training'></a>

In [ ]:
# ── Per-segment tail split ────────────────────────────────────────────────
# Each segment independently contributes its last VAL_STEPS rows to val
# and everything before to train. Avoids the time-window bias of a global
# timestamp cut (which would put all segments' evening data in val).

VAL_STEPS = 20

train_rows, val_rows = [], []
for link_id, grp in df_all_sorted.groupby('LinkID', sort=False):
    grp = grp.sort_values('timestamp')
    if len(grp) <= VAL_STEPS:
        train_rows.append(grp)
        continue
    train_rows.append(grp.iloc[:-VAL_STEPS])
    val_rows.append(grp.iloc[-VAL_STEPS:])

df_train_raw = pd.concat(train_rows, ignore_index=True)
df_val_raw   = pd.concat(val_rows,   ignore_index=True)

print(f'Train rows: {len(df_train_raw):,} | Val rows: {len(df_val_raw):,}')
print(f'Train segments: {df_train_raw["LinkID"].nunique()} | '
      f'Val segments: {df_val_raw["LinkID"].nunique()}')


# ── Scaler fitted on ALL data ─────────────────────────────────────────────
# Fitting on train-only contaminates the scale with whichever time window
# happened to be collected first. The scaler's job is unit normalisation,
# not learning what's "normal" — that's the autoencoder's job.

full_scaler = StandardScaler()
full_X = df_all_sorted[FEATURE_COLS].fillna(0).values.astype(np.float32)
full_scaler.fit(full_X)
print(f'Scaler fit on {len(full_X):,} rows (full dataset)')


# ── Flat dataset (one row = one sample, no sequences) ────────────────────
class FlatFeatureDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_cols: list,
                 scaler: StandardScaler, normal_only: bool = False):

        if normal_only and 'is_anomaly' in df.columns:
            df = df[~df['is_anomaly']]

        for c in feature_cols:
            if c not in df.columns:
                df[c] = 0.0

        X = df[feature_cols].fillna(0).values.astype(np.float32)
        X = scaler.transform(X)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

        # Clip to 5 std — removes extreme outliers without losing signal
        train_std  = X.std()
        clip_val   = min(train_std * 5, 5.0)
        X          = np.clip(X, -clip_val, clip_val)

        self.X      = torch.tensor(X, dtype=torch.float32)
        self.labels = df['is_anomaly'].tolist() if 'is_anomaly' in df.columns else [False] * len(df)

    def __len__(self):          return len(self.X)
    def __getitem__(self, i):   return self.X[i]


train_dataset = FlatFeatureDataset(df_train_raw, FEATURE_COLS, scaler=full_scaler)
val_dataset   = FlatFeatureDataset(df_val_raw,   FEATURE_COLS, scaler=full_scaler)
scaler        = full_scaler   # keep 'scaler' name for downstream inference cells

print(f'Train samples: {len(train_dataset):,} | Val samples: {len(val_dataset):,}')
if len(set(val_dataset.labels)) > 1:
    print(f'Val anomaly rate: {np.mean(val_dataset.labels):.2%}')

# Quick distribution sanity check
print(f'\nTrain scaled — mean: {train_dataset.X.mean():.3f}  std: {train_dataset.X.std():.3f}')
print(f'Val scaled   — mean: {val_dataset.X.mean():.3f}  std: {val_dataset.X.std():.3f}')


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────

def train_autoencoder(
        model:      FeedforwardAutoencoder,
        train_data: FlatFeatureDataset,
        val_data:   FlatFeatureDataset,
        n_epochs:   int   = 60,
        lr:         float = 1e-3,
        batch_size: int   = 2048,
        patience:   int   = 8,
) -> dict:

    if len(train_data) == 0:
        print('No training data.')
        return {}

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(val_data,   batch_size=2048, shuffle=False,
                              num_workers=0)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-6
    )
    criterion = nn.MSELoss()

    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_val_loss = float('inf')
    patience_ctr  = 0
    best_state    = None

    for epoch in range(1, n_epochs + 1):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            recon = model(batch)
            loss  = criterion(recon, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        val_losses, all_errors = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE)
                recon = model(batch)
                val_losses.append(criterion(recon, batch).item())
                all_errors.extend(
                    ((batch - recon) ** 2).mean(dim=1).cpu().tolist()
                )

        t_loss = np.mean(train_losses)
        v_loss = np.mean(val_losses)
        scheduler.step()
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)

        auc_str = ''
        if len(set(val_data.labels)) > 1:
            try:
                auc = roc_auc_score(val_data.labels, all_errors)
                history['val_auc'].append(auc)
                auc_str = f' | AUC={auc:.3f}'
            except:
                history['val_auc'].append(0.5)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            patience_ctr  = 0
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{n_epochs} | '
                  f'Train: {t_loss:.5f} | Val: {v_loss:.5f}{auc_str}')

        if patience_ctr >= patience:
            print(f'Early stopping at epoch {epoch}')
            break

    if best_state:
        model.load_state_dict(best_state)
    print(f'\nTraining complete. Best val loss: {best_val_loss:.5f}')
    return history


# ── Run ───────────────────────────────────────────────────────────────────
history = train_autoencoder(
    model, train_dataset, val_dataset,
    n_epochs=100, lr=5e-4, batch_size=512, patience=15
)


In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────
if history:
    fig, axes = plt.subplots(1, 2 if history.get('val_auc') else 1, figsize=(14, 4))
    axes = [axes] if not isinstance(axes, np.ndarray) else axes

    axes[0].plot(history['train_loss'], label='Train Loss', color='steelblue')
    axes[0].plot(history['val_loss'],   label='Val Loss',   color='coral')
    axes[0].set_title('LSTM Autoencoder — Reconstruction Loss (MSE)', fontsize=13)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    if history.get('val_auc') and len(axes) > 1:
        axes[1].plot(history['val_auc'], color='mediumseagreen', label='Val AUC')
        axes[1].axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='Random')
        axes[1].set_title('Anomaly Detection AUC (Val Set)', fontsize=13)
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC AUC')
        axes[1].legend(); axes[1].grid(alpha=0.3)
        axes[1].set_ylim([0.4, 1.0])

    plt.tight_layout()
    plt.savefig(DATA_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Training curves saved.')


In [ ]:
# ── Save model + scaler ───────────────────────────────────────────────────
import pickle

MODEL_PATH  = DATA_DIR / 'lstm_autoencoder.pt'
SCALER_PATH = DATA_DIR / 'scaler.pkl'

#torch.save(model.state_dict(), MODEL_PATH)
#with open(SCALER_PATH, 'wb') as f:
    #pickle.dump(scaler, f)

#print(f'Model  saved → {MODEL_PATH}')
#print(f'Scaler saved → {SCALER_PATH}')

# ── Load back (example) ───────────────────────────────────────────────────
def load_model_and_scaler():
    m = FeedforwardAutoencoder(
        feature_dim=FEATURE_DIM,
    ).to(DEVICE)
    m.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    m.eval()
    with open(SCALER_PATH, 'rb') as f:
        sc = pickle.load(f)
    return m, sc

print('Use load_model_and_scaler() to reload for inference.')


## 7. Risk Index Inference <a id='7-inference'></a>

Converts raw reconstruction error into an interpretable **Risk Index [0–100]**:

1. **Anomaly score** = z-score of reconstruction error relative to per-(segment, hour) rolling baseline
2. **Domain multipliers** applied for severe known conditions (floods, accidents, blackouts)
3. **Sigmoid squashing** maps to [0, 100]


In [ ]:
# ── Baseline statistics: per (link_id, hour_of_day) rolling statistics ────

class BaselineTracker:
    """
    Maintains running mean and std of reconstruction errors
    per (link_id, hour_of_day) bin.
    Falls back to global statistics when per-segment bins are sparse.
    """
    def __init__(self):
        self._bins   = defaultdict(lambda: deque(maxlen=500))
        self._global = deque(maxlen=10000)

    def update(self, link_id: str, hour: int, error: float):
        self._bins[(link_id, hour)].append(error)
        self._global.append(error)

    def get_stats(self, link_id: str, hour: int, min_samples: int = 10):
        bin_data = list(self._bins[(link_id, hour)])
        if len(bin_data) >= min_samples:
            return np.mean(bin_data), max(np.std(bin_data), 1e-6)
        global_data = list(self._global)
        if len(global_data) >= min_samples:
            return np.mean(global_data), max(np.std(global_data), 1e-6)
        return 0.5, 0.5

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump(dict(bins=dict(self._bins), global_=list(self._global)), f)

    @classmethod
    def load(cls, path):
        bt = cls()
        with open(path, 'rb') as f:
            d = pickle.load(f)
        for k, v in d['bins'].items():
            bt._bins[k] = deque(v, maxlen=500)
        bt._global = deque(d['global_'], maxlen=10000)
        return bt


# ── Warm up baseline tracker on training data ─────────────────────────────
# FlatFeatureDataset preserves df_train_raw row order, so we can zip
# reconstruction errors directly against the source DataFrame's metadata.

baseline = BaselineTracker()
model.eval()

# Score all training rows in large batches
train_errors = []
loader = DataLoader(train_dataset, batch_size=2048, shuffle=False)
with torch.no_grad():
    for batch in loader:
        errs = model.reconstruction_error(batch.to(DEVICE)).cpu().numpy()
        train_errors.extend(errs.tolist())

# df_train_raw is aligned with train_dataset (same order, no shuffling)
# so zip directly — no index arithmetic needed
assert len(train_errors) == len(df_train_raw), \
    f'Length mismatch: {len(train_errors)} errors vs {len(df_train_raw)} rows'

for err, (_, row) in zip(train_errors, df_train_raw.iterrows()):
    baseline.update(str(row['LinkID']), pd.Timestamp(row['timestamp']).hour, err)

baseline.save(DATA_DIR / 'baseline_tracker.pkl')

global_errors = list(baseline._global)
print(f'Baseline tracker populated.')
print(f'  Bins (segment, hour): {len(baseline._bins):,}')
print(f'  Global samples:       {len(global_errors):,}')
print(f'  Mean:  {np.mean(global_errors):.4f}')
print(f'  Std:   {np.std(global_errors):.4f}')
print(f'  p50:   {np.percentile(global_errors, 50):.4f}')
print(f'  p95:   {np.percentile(global_errors, 95):.4f}')
print(f'  p99:   {np.percentile(global_errors, 99):.4f}')

In [ ]:
# ── Risk Index computation ────────────────────────────────────────────────

def sigmoid(x):
    # Clamp to [-500, 500] — sigmoid is numerically 0 or 1 beyond ±20 anyway
    x = max(-500.0, min(500.0, x))
    return 1.0 / (1.0 + math.exp(-x))





def compute_domain_multiplier(feature_row: dict) -> float:
    """
    Computes the product of applicable domain multipliers from the raw feature row.
    These are hard-coded domain-knowledge boosts for known severe conditions.
    """
    m = 1.0
    if feature_row.get('flag_accident'):            m *= DOMAIN_MULTIPLIERS['active_accident']
    if feature_row.get('flood_severity_score', 0) >= 4: m *= DOMAIN_MULTIPLIERS['flood_extreme']
    elif feature_row.get('flood_severity_score', 0) >= 3: m *= DOMAIN_MULTIPLIERS['flood_severe']
    elif feature_row.get('flood_severity_score', 0) >= 2: m *= DOMAIN_MULTIPLIERS['flood_moderate']
    elif feature_row.get('flood_severity_score', 0) >= 1: m *= DOMAIN_MULTIPLIERS['flood_minor']
    if feature_row.get('blackout_count', 0) > 0:   m *= DOMAIN_MULTIPLIERS['faulty_light_blackout']
    elif feature_row.get('flashing_count', 0) > 0: m *= DOMAIN_MULTIPLIERS['faulty_light_flashing']
    if feature_row.get('is_late_night'):            m *= DOMAIN_MULTIPLIERS['late_night']
    elif feature_row.get('is_peak_hour'):           m *= DOMAIN_MULTIPLIERS['peak_hour']
    if feature_row.get('flag_roadwork'):            m *= DOMAIN_MULTIPLIERS['active_roadworks']
    if feature_row.get('emas_breakdown_score', 0) > 1: m *= DOMAIN_MULTIPLIERS['emas_breakdown']
    if feature_row.get('emas_rain_score', 0) > 1:  m *= DOMAIN_MULTIPLIERS['emas_rain']
    return m


RISK_SCALE_FACTOR = 3.0 

def reconstruction_error_to_risk(error: float, link_id: str, hour: int,
                                  feature_row: dict,
                                  baseline_tracker: BaselineTracker) -> float:
    mu, sigma = baseline_tracker.get_stats(link_id, hour)
    
    # Guard: if sigma is near-zero (flat baseline), the z-score explodes.
    # Use a minimum sigma of 10% of mu, floored at 1e-3.
    sigma = max(sigma, max(mu * 0.1, 1e-3))
    
    z_score  = (error - mu) / sigma
    dm       = compute_domain_multiplier(feature_row)
    
    # Log-transform the error before z-scoring to handle right-skewed MSE distribution
    log_error = math.log1p(error)
    log_mu, log_sigma = math.log1p(mu), max(math.log1p(mu + sigma) - math.log1p(mu), 1e-3)
    z_score   = (log_error - log_mu) / log_sigma   # replace raw z-score with log-space z-score
    
    raw_risk  = np.clip(z_score * dm / RISK_SCALE_FACTOR, -6.0, 6.0)  # ±6 gives full 0–100 range
    return round(100.0 * sigmoid(raw_risk), 2)

print('Risk index computation functions defined.')

# Updated unit test — should now show spread across 0–100, not clustered at 50
_test_row = {'flag_accident': 0, 'flood_severity_score': 0,
              'blackout_count': 0, 'flashing_count': 0,
              'is_late_night': 0, 'is_peak_hour': 1,
              'flag_roadwork': 0, 'emas_breakdown_score': 0, 'emas_rain_score': 0}
print('Normal conditions (peak hour only):')
for _err in [0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]:
    ri = reconstruction_error_to_risk(_err, 'TEST', 8, _test_row, baseline)
    print(f'  error={_err:.3f} → risk_index={ri:.1f}')

# Also test with accident flag — should push scores noticeably higher
_accident_row = {**_test_row, 'flag_accident': 1}
print('\nWith active accident:')
for _err in [0.1, 0.5, 1.0]:
    ri = reconstruction_error_to_risk(_err, 'TEST', 8, _accident_row, baseline)
    print(f'  error={_err:.3f} → risk_index={ri:.1f}')

In [ ]:
# ── Batch inference: score the entire validation set ─────────────────────

def score_dataset(model, dataset: FlatFeatureDataset,
                  baseline_tracker: BaselineTracker,
                  df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Runs inference on a dataset and returns a DataFrame with:
      link_id, timestamp, reconstruction_error, risk_index,
      and optionally is_anomaly (ground truth) for evaluation.

    df_raw must be the same DataFrame used to build dataset (same row order).
    """
    model.eval()

    # Score all rows in one pass
    all_errors = []
    loader = DataLoader(dataset, batch_size=2048, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            errs = model.reconstruction_error(batch.to(DEVICE)).cpu().numpy()
            all_errors.extend(errs.tolist())

    assert len(all_errors) == len(df_raw), \
        f'Length mismatch: {len(all_errors)} errors vs {len(df_raw)} rows'

    # Compute risk index vectorised where possible, then assemble output
    errors    = np.array(all_errors)
    link_ids  = df_raw['LinkID'].values
    hours     = pd.to_datetime(df_raw['timestamp']).dt.hour.values
    labels    = dataset.labels

    risk_indices = np.zeros(len(errors))
    for i, (err, lid, hour) in enumerate(zip(errors, link_ids, hours)):
        feat_row = df_raw.iloc[i].to_dict()
        risk_indices[i] = reconstruction_error_to_risk(
            float(err), str(lid), int(hour), feat_row, baseline_tracker
        )
        baseline_tracker.update(str(lid), int(hour), float(err))

    df_scores = df_raw[['LinkID', 'timestamp']].copy()
    df_scores['reconstruction_error'] = errors
    df_scores['risk_index']           = risk_indices
    df_scores['is_anomaly']           = labels

    # Carry through useful display columns if present
    for col in ['road_name', 'speed_band', 'seg_lat', 'seg_lon']:
        if col in df_raw.columns:
            df_scores[col] = df_raw[col].values

    return df_scores.reset_index(drop=True)


print('Running inference on validation set...')
df_scores = score_dataset(model, val_dataset, baseline, df_val_raw)
print(f'Scored {len(df_scores):,} rows.')
print(df_scores[['reconstruction_error', 'risk_index']].describe().round(3))

In [ ]:
load_model_and_scaler()

In [ ]:
# ── Evaluation: AUC, threshold analysis ──────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

if True:
    print('No anomaly labels in val set — showing risk index distribution only.')
    df_scores['risk_index'] = df_scores['risk_index'] - df_scores['risk_index'].min()

    df_scores['risk_index'].hist(bins=40, figsize=(8,4), color='steelblue', alpha=0.7)
    plt.title('Risk Index Distribution (Val Set)')
    plt.xlabel('Risk Index [0–100]'); plt.ylabel('Count')
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION WITHOUT GROUND-TRUTH LABELS
# ═══════════════════════════════════════════════════════════════════════════

# ── 1. SYNTHETIC ANOMALY INJECTION ────────────────────────────────────────
def inject_anomalies(df: pd.DataFrame, fraction: float = 0.05, seed: int = 42) -> pd.DataFrame:
    """Corrupt a fraction of rows with traffic-plausible anomalies."""
    rng = np.random.default_rng(seed)
    df_syn = df.copy()
    n = int(len(df_syn) * fraction)
    idx = rng.choice(len(df_syn), size=n, replace=False)

    # Speed collapse (gridlock / incident)
    if 'speed' in df_syn.columns:
        df_syn.iloc[idx, df_syn.columns.get_loc('speed')] *= rng.uniform(0.05, 0.25, size=n)

    # speed_drop_rate spike — the key engineered feature
    if 'speed_drop_rate' in df_syn.columns:
        df_syn.iloc[idx, df_syn.columns.get_loc('speed_drop_rate')] += rng.uniform(3.0, 7.0, size=n)

    # travel_time blowup
    if 'travel_time' in df_syn.columns:
        df_syn.iloc[idx, df_syn.columns.get_loc('travel_time')] *= rng.uniform(3.0, 8.0, size=n)

    df_syn['is_anomaly'] = False
    df_syn.iloc[idx, df_syn.columns.get_loc('is_anomaly')] = True
    return df_syn

# Score synthetic val set
df_val_syn = inject_anomalies(df_val_raw, fraction=0.05)
val_syn_dataset = FlatFeatureDataset(df_val_syn, FEATURE_COLS, scaler=full_scaler)
val_syn_loader  = DataLoader(val_syn_dataset, batch_size=2048, shuffle=False)

model.eval()
syn_errors = []
with torch.no_grad():
    for batch in val_syn_loader:
        batch = batch.to(DEVICE)
        syn_errors.extend(model.reconstruction_error(batch).cpu().tolist())

syn_errors = np.array(syn_errors)
y_true_syn = np.array(val_syn_dataset.labels).astype(int)

auc_roc_syn = roc_auc_score(y_true_syn, syn_errors)
auc_pr_syn  = average_precision_score(y_true_syn, syn_errors)
print(f'[Synthetic] ROC AUC = {auc_roc_syn:.4f}  |  PR AUC = {auc_pr_syn:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fpr, tpr, _ = roc_curve(y_true_syn, syn_errors)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC={auc_roc_syn:.3f}')
axes[0].plot([0,1],[0,1],'--', color='gray')
axes[0].set_title('ROC Curve (Synthetic Labels)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(alpha=0.3)

prec, rec, _ = precision_recall_curve(y_true_syn, syn_errors)
axes[1].plot(rec, prec, color='coral', lw=2, label=f'AP={auc_pr_syn:.3f}')
axes[1].set_title('PR Curve (Synthetic Labels)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].hist(syn_errors[y_true_syn == 0], bins=40, alpha=0.6, color='steelblue',
             label='Normal', density=True)
axes[2].hist(syn_errors[y_true_syn == 1], bins=40, alpha=0.6, color='crimson',
             label='Synthetic Anomaly', density=True)
axes[2].set_title('Reconstruction Error Distribution')
axes[2].set_xlabel('MSE'); axes[2].set_ylabel('Density')
axes[2].legend(); axes[2].grid(alpha=0.3)
plt.suptitle('Evaluation via Synthetic Anomaly Injection', y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'eval_synthetic.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3. FLAG RATE + TEMPORAL COHERENCE ─────────────────────────────────────
# Assumes df_scores already exists from your scoring cell (with 'risk_index',
# 'timestamp', 'LinkID' columns). If not, build it from df_val_raw below.


# ---- Flag rate --------------------------------------------------------
for thresh in [50, 60, 70, 80, 90]:
    flagged    = (df_scores['risk_index'] >= thresh).sum()
    flag_rate  = flagged / len(df_scores)
    print(f'Threshold {thresh:3d} | Flagged: {flagged:5d} / {len(df_scores):,} '
          f'({flag_rate:.2%})')
# Target: 1–10% flags. <0.1% → model too conservative. >20% → poor discrimination.

# ---- Temporal coherence -----------------------------------------------
# Anomalies should cluster in time — compute lag-1 autocorrelation of flag series.
df_scores_eval = df_scores.sort_values('timestamp')
high_risk_series = (df_scores_eval['risk_index'] >= 80).astype(float).values
acf_lag1 = np.corrcoef(high_risk_series[:-1], high_risk_series[1:])[0, 1]
print(f'\nFlag autocorrelation (lag-1): {acf_lag1:.4f}')
print('  > 0.1  → flags cluster in time (good — real congestion persists)')
print('  ≈ 0.0  → flags are random noise (bad)')

# ---- Visual: risk index over time per segment -------------------------
fig, axes = plt.subplots(len(plot_seg_ids), 1,
                         figsize=(14, 3 * len(plot_seg_ids)), squeeze=False)
for i, link_id in enumerate(plot_seg_ids):
    ax   = axes[i][0]
    grp  = df_scores_eval[df_scores_eval['LinkID'] == link_id].sort_values('timestamp')
    ax.plot(grp['timestamp'], grp['risk_index'], color='steelblue', lw=1.5)
    ax.axhline(80, color='crimson', linestyle='--', lw=1, label='Threshold (80)')
    ax.fill_between(grp['timestamp'], 0, grp['risk_index'],
                    where=grp['risk_index'] >= 80,
                    color='crimson', alpha=0.3, label='Flagged')
    ax.set_title(f'LinkID {link_id} — Risk Index over Time', fontsize=9)
    ax.set_ylabel('Risk Index'); ax.set_ylim(0, 105)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Temporal Coherence of Flags (Val Set)', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / 'eval_temporal.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Top risky segments at latest snapshot ────────────────────────────────

def get_latest_risk_snapshot(df_scores: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """Returns the most recent risk score for each segment, sorted by risk."""
    latest = (
        df_scores
        .sort_values('timestamp')
        .groupby('link_id')
        .last()
        .reset_index()
        .sort_values('risk_index', ascending=False)
    )
    return latest.head(top_n)


top_risky = get_latest_risk_snapshot(df_scores, top_n=20)
print('Top 20 highest-risk segments:')
display_cols = ['link_id', 'risk_index', 'reconstruction_error', 'speed_band',
                'road_name', 'timestamp']
display_cols = [c for c in display_cols if c in top_risky.columns]
print(top_risky[display_cols].to_string(index=False))


In [ ]:
# ── Static heatmap: risk index over time for top segments ────────────────

def plot_risk_heatmap(df_scores: pd.DataFrame, n_segments: int = 30,
                      figsize=(16, 8)):
    """
    Heatmap of risk_index[segment × time] for the top-N highest-risk segments.
    Rows = segments, columns = timestep buckets.
    """
    top_segs = (
        df_scores.groupby('link_id')['risk_index'].mean()
        .nlargest(n_segments).index.tolist()
    )
    df_sub = df_scores[df_scores['link_id'].isin(top_segs)].copy()
    df_sub['time_bucket'] = pd.to_datetime(df_sub['timestamp']).dt.floor('10min')

    pivot = df_sub.pivot_table(
        index='link_id', columns='time_bucket', values='risk_index',
        aggfunc='mean'
    ).fillna(0)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
                   vmin=0, vmax=100, interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Risk Index [0–100]')

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=7)
    xtick_step = max(1, len(pivot.columns)//10)
    ax.set_xticks(range(0, len(pivot.columns), xtick_step))
    ax.set_xticklabels(
        [str(t)[-8:-3] for t in pivot.columns[::xtick_step]],
        rotation=45, ha='right', fontsize=8
    )
    ax.set_title(f'Road Risk Index Heatmap — Top {n_segments} Segments', fontsize=14)
    ax.set_xlabel('Time'); ax.set_ylabel('Segment ID')
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'risk_heatmap.png', dpi=120, bbox_inches='tight')
    plt.show()


plot_risk_heatmap(df_scores, n_segments=25)


In [ ]:
# ── Geospatial risk map (static matplotlib scatter) ───────────────────────

def plot_risk_map(df_scores: pd.DataFrame, figsize=(12, 10)):
    """
    Scatter plot of Singapore road segments coloured by current risk index.
    Green = low risk, Yellow = medium, Red = high risk.
    """
    latest = (
        df_scores.sort_values('timestamp')
        .groupby('link_id').last().reset_index()
    )
    # Filter to Singapore bounding box
    latest = latest[
        (latest['seg_lat'].between(1.15, 1.50)) &
        (latest['seg_lon'].between(103.55, 104.10))
    ].dropna(subset=['seg_lat', 'seg_lon', 'risk_index'])

    if latest.empty:
        print('No geospatial data available.')
        return

    fig, ax = plt.subplots(figsize=figsize, facecolor='#1a1a2e')
    ax.set_facecolor('#1a1a2e')

    sc = ax.scatter(
        latest['seg_lon'], latest['seg_lat'],
        c=latest['risk_index'],
        cmap='RdYlGn_r', vmin=0, vmax=100,
        s=8, alpha=0.75, linewidths=0
    )
    cbar = plt.colorbar(sc, ax=ax, shrink=0.7)
    cbar.set_label('Risk Index [0–100]', color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

    # Highlight high-risk segments
    high_risk = latest[latest['risk_index'] >= 70]
    if not high_risk.empty:
        ax.scatter(high_risk['seg_lon'], high_risk['seg_lat'],
                   s=40, c='red', marker='*', zorder=5, label='High Risk (≥70)',
                   edgecolors='white', linewidths=0.5)
        ax.legend(loc='lower right', facecolor='#2a2a4e', edgecolor='white',
                  labelcolor='white', fontsize=9)

    ax.set_title('🚦 Singapore Road Risk Index Map',
                 color='white', fontsize=15, pad=12)
    ax.set_xlabel('Longitude', color='white')
    ax.set_ylabel('Latitude',  color='white')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444488')

    ts_str = str(latest['timestamp'].max())[:16]
    ax.text(0.02, 0.02, f'Updated: {ts_str}', transform=ax.transAxes,
            color='#aaaaaa', fontsize=8)

    plt.tight_layout()
    plt.savefig(DATA_DIR / 'risk_map.png', dpi=150, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.show()
    print(f'Risk map saved. Showing {len(latest)} segments.')


plot_risk_map(df_scores)


In [ ]:
# ── Risk index time-series for individual segments ────────────────────────

def plot_segment_timeseries(df_scores: pd.DataFrame,
                             link_ids: list = None, n_top: int = 5):
    """
    Plot risk index over time for selected (or top-N) segments.
    Shows ground-truth anomaly markers when available.
    """
    if link_ids is None:
        link_ids = (
            df_scores.groupby('link_id')['risk_index'].mean()
            .nlargest(n_top).index.tolist()
        )

    fig, axes = plt.subplots(len(link_ids), 1,
                              figsize=(14, 3*len(link_ids)),
                              sharex=False)
    if len(link_ids) == 1:
        axes = [axes]

    colors = plt.cm.tab10.colors
    for i, (lid, ax) in enumerate(zip(link_ids, axes)):
        sub = df_scores[df_scores['link_id'] == lid].sort_values('timestamp')
        ax.plot(sub['timestamp'], sub['risk_index'],
                color=colors[i % 10], lw=1.5, label='Risk Index')
        ax.fill_between(sub['timestamp'], sub['risk_index'],
                        alpha=0.15, color=colors[i % 10])

        # Threshold lines
        ax.axhline(70, color='red',    lw=0.8, ls='--', alpha=0.6, label='High (70)')
        ax.axhline(45, color='orange', lw=0.8, ls='--', alpha=0.6, label='Med (45)')

        # Anomaly ground truth markers
        if 'is_anomaly' in sub.columns:
            anom = sub[sub['is_anomaly'] == True]
            if not anom.empty:
                ax.scatter(anom['timestamp'], anom['risk_index'],
                           color='red', s=40, zorder=5, marker='x',
                           label='True Anomaly', linewidths=1.5)

        road_name = sub['road_name'].iloc[0] if 'road_name' in sub.columns else ''
        ax.set_title(f'Segment {lid}  |  {road_name}', fontsize=10)
        ax.set_ylabel('Risk Index'); ax.set_ylim([0, 105])
        ax.legend(fontsize=7, loc='upper right'); ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig(DATA_DIR / 'segment_timeseries.png', dpi=120, bbox_inches='tight')
    plt.show()


plot_segment_timeseries(df_scores, n_top=5)


In [ ]:
# ── Segment sequence buffer: maintains rolling window per link_id ─────────

class SegmentSequenceBuffer:
    """
    Maintains a rolling buffer of the last SEQUENCE_LENGTH feature vectors
    for each road segment. When the buffer is full, it provides a ready-to-score
    sequence tensor.
    """
    def __init__(self, seq_len: int = SEQUENCE_LENGTH, feature_dim: int = FEATURE_DIM):
        self.seq_len     = seq_len
        self.feature_dim = feature_dim
        self._buffers    = defaultdict(lambda: deque(maxlen=seq_len))

    def push(self, link_id: str, feature_vec: np.ndarray):
        """Add a new feature vector for a segment."""
        self._buffers[link_id].append(feature_vec.copy())

    def get_sequence(self, link_id: str):
        """
        Returns a (seq_len, feature_dim) numpy array if the buffer is full,
        else None.
        """
        buf = self._buffers[link_id]
        if len(buf) < self.seq_len:
            return None
        return np.stack(list(buf), axis=0).astype(np.float32)

    def has_sequence(self, link_id: str) -> bool:
        return len(self._buffers[link_id]) >= self.seq_len

    def n_ready(self) -> int:
        return sum(1 for buf in self._buffers.values() if len(buf) >= self.seq_len)


seq_buffer = SegmentSequenceBuffer(SEQUENCE_LENGTH, FEATURE_DIM)
print(f'SegmentSequenceBuffer initialised (seq_len={SEQUENCE_LENGTH}, feat_dim={FEATURE_DIM})')


---
## Summary

| Component | Details |
|---|---|
| **APIs used** | Traffic Speed Bands, Traffic Incidents, Faulty Traffic Lights, VMS/EMAS, Flood Alerts |
| **Feature dim** | 30 (speed × 5, incidents × 7, lights × 3, VMS × 2, flood × 2, road_cat × 7, time × 6) |
| **Model** | BiLSTM Encoder + LSTM Decoder, ~80K params |
| **Sequence length** | 12 timesteps × 2 min = 24-min look-back |
| **Anomaly detection** | Reconstruction error z-scored per (segment, hour) baseline |
| **Risk index** | Z-score × domain multipliers → sigmoid → [0, 100] |
| **Update frequency** | Every 2 minutes (fastest API refresh) |
| **Self-improving** | Retrains every 300 new samples using pseudo-labels from risk_index > 70 |

### Next Steps
- **Get a real API key** from [LTA DataMall](https://datamall.lta.gov.sg) and run `run_collection()` for 2+ hours
- **Switch `USING_REAL_DATA = True`** and retrain on real data
- **Run `run_live()`** to start the continuous monitoring system
- **Accumulate `Traffic Incidents` labels** over time for semi-supervised fine-tuning (Section 4 → Phase 3)
